In [ ]:
dbutils.widgets.dropdown(
    "run_fixture_tests",
    "false",
    ["false", "true"]
)

run_fixture_tests = (
    dbutils.widgets.get("run_fixture_tests") == "true"
)

In [ ]:
import re

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")

catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name.")

# Create a volume for incoming supplier files.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{catalog}.bronze.supplier_files"
)

# Use a dedicated directory for product files.
product_source_path = (
    f"/Volumes/{catalog}/bronze/supplier_files/products"
)

dbutils.fs.mkdirs(product_source_path)

print("Upload the product CSV into:")
print(product_source_path)

In [ ]:
uploaded_files = dbutils.fs.ls(product_source_path)

for file_info in uploaded_files:
    print(
        f"File: {file_info.name} | "
        f"Size: {file_info.size} bytes"
    )

if run_fixture_tests:
    assert any(
        file_info.name == "products_batch_001.csv"
        for file_info in uploaded_files
    ), "products_batch_001.csv was not found."

    print("Original product seed file exists.")

else:
    print("Skipped the original product seed-file check.")

In [ ]:
from pyspark.sql import types as T

# Keep source values as strings in Bronze.
product_raw_schema = T.StructType([
    T.StructField("product_id", T.StringType(), True),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("catalog_price", T.StringType(), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("is_active", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("_rescued_data", T.StringType(), True),
])

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.operations"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{catalog}.operations.streaming_checkpoints"
)

product_state_path = (
    f"/Volumes/{catalog}/operations/streaming_checkpoints/"
    "product_catalog_v1"
)

product_schema_path = f"{product_state_path}/schema"
product_checkpoint_path = f"{product_state_path}/checkpoint"

product_bronze_table = f"{catalog}.bronze.products_raw"

In [ ]:
product_stream = (
    spark.readStream
    .format("cloudFiles")
    .option('cloudFiles.format' , "csv")
    .option("header" , "true")
    .option("pathGlobFilter", "*.csv")
    .option('cloudFiles.schemaLocation' , product_schema_path)
    .option("cloudFiles.schemaEvolutionMode" , "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(product_raw_schema)
    .load(product_source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

print("Streaming DataFrame:", product_stream.isStreaming)

In [ ]:
product_query = (
    product_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation" , product_checkpoint_path)
    .trigger(availableNow = True)
    .toTable(product_bronze_table)
)

product_query.awaitTermination()

print(f"Product ingestion completed: {product_bronze_table}")

In [ ]:
saved_products = spark.table(product_bronze_table)

display(
    saved_products.orderBy("product_id")
)

display(
    saved_products
    .groupBy("source_file")
    .count()
)

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")


parsed_products = (
    spark.table(product_bronze_table)
    .withColumnRenamed("catalog_price" , "catalog_price_raw")
    .withColumnRenamed("is_active" , "is_active_raw")
    .withColumnRenamed("updated_at" , "updated_at_raw")
    .withColumn(
        'catalog_price',
        F.expr("try_cast(catalog_price_raw AS DECIMAL(18,2))")
    )
    .withColumn(
        "is_active",
        F.when(
            F.lower(F.trim(F.col("is_active_raw"))) == "true",
            F.lit(True)
        )
        .when(
            F.lower(F.trim(F.col("is_active_raw"))) == "false",
            F.lit(False)
        )
        .otherwise(
            F.lit(None).cast("boolean")
        )
    )
    .withColumn(
        "updated_at",
        F.try_to_timestamp("updated_at_raw")
    )
)
display(parsed_products)

In [ ]:
def has_text(column):
    return F.coalesce(
        F.length(F.trim(column)) > 0,
        F.lit(False)
    )


product_rules = [
    (
        has_text(F.col("product_id")),
        "Missing product_id"
    ),
    (
        has_text(F.col("product_name")),
        'Missing product_name'
    ),
    (
        has_text(F.col("category")),
        "Missing category"
    ),
    (
        F.col('currency') == "GBP",
        "Missing or unsupported currency"
    ),
    (
        F.col("catalog_price_raw").rlike(
            r"^[0-9]+(\.[0-9]{1,2})?$"
        ),
        "Price must contain digits with at most two decimal places"
    ),
    (
        F.col("catalog_price").isNotNull()
        & (F.col("catalog_price") > 0),
        "Price must be positive and fit DECIMAL(12, 2)"
    ),
    (
        F.col("is_active").isNotNull(),
        "is_active must be true or false"
    ),
     (
        F.col("updated_at").isNotNull(),
        "Missing or invalid updated_at"
    ),
    (
        F.col("_rescued_data").isNull(),
        "Unexpected data was rescued during ingestion"
    ),
]


checked_products = (
    parsed_products
    .withColumn(
        "rejection_reasons",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition , reason in product_rules
            ]),
            lambda reason : reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        F.size("rejection_reasons") == 0
    )
)

In [ ]:
valid_products = checked_products.filter(
    F.col("is_valid")
)

quarantine_products = checked_products.filter(
    ~F.col("is_valid")
)

display(
    checked_products
    .groupBy("is_valid")
    .count()
)

display(
    quarantine_products.select(
        "product_id",
        "catalog_price_raw",
        "is_active_raw",
        "updated_at_raw",
        "_rescued_data",
        "rejection_reasons",
        "source_file",
    )
)

In [ ]:
from pyspark.sql.window import Window

# Create a consistent representation of each product's attributes.
product_candidates = valid_products.withColumn(
    "product_payload",
    F.to_json(
        F.struct(
            "product_name",
            "category",
            "catalog_price",
            "currency",
            "is_active",
        ),
        options={"ignoreNullFields": "false"}
    )
)

# Detect conflicting contents for the same product version.
product_version_checks = (
    product_candidates
    .groupBy("product_id", "updated_at")
    .agg(
        F.countDistinct("product_payload").alias("payload_versions")
    )
)

product_conflicts = product_version_checks.filter(
    F.col("payload_versions") > 1
)

if product_conflicts.limit(1).count() > 0:
    display(product_conflicts)

    raise ValueError(
        "Conflicting records share the same product_id and updated_at."
    )

# Rank each product's versions, newest first.
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("ingested_at").asc(),
        F.col("source_file").asc(),
    )
)

current_products = (
    product_candidates
    .withColumn(
        "version_rank",
        F.row_number().over(product_window)
    )
    .filter(F.col("version_rank") == 1)
    .select(
        "product_id",
        "product_name",
        "category",
        "catalog_price",
        "currency",
        "is_active",
        "updated_at",
        "source_file",
        "ingested_at",
    )
)

display(current_products.orderBy("product_id"))

In [ ]:
from delta.tables import DeltaTable

product_silver_table = f"{catalog}.silver.products_current"

current_products.createOrReplaceTempView(
    'current_products_source'
)

# Create an empty destination if needed.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {product_silver_table}
    USING DELTA 
    AS 
    SELECT * 
    FROM current_products_source
    Where 1 = 0      
""")

# Synchronize from the complete accepted product history.

(
    DeltaTable.forName(spark, product_silver_table)
    .alias("target")
    .merge(
        current_products.alias("source"),
        "target.product_id = source.product_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {product_silver_table}")

In [ ]:
saved_current_products = spark.table(product_silver_table)

summary_df = saved_current_products.agg(
    F.count("*").alias("saved_products"),
    F.countDistinct("product_id").alias("unique_product_ids"),
)

display(summary_df)

summary = summary_df.first()

assert summary["saved_products"] == summary["unique_product_ids"], (
    "Duplicate product IDs exist in the Silver catalogue."
)

assert summary["saved_products"] == current_products.count(), (
    "The saved product count differs from the prepared source."
)

display(
    saved_current_products.orderBy("product_id")
)

print("Current-product checks passed.")

In [ ]:
if run_fixture_tests:
    from decimal import Decimal

    # Check the catalogue value from our original update test.
    catalog_product = (
        spark.table(f"{catalog}.silver.products_current")
        .filter(F.col("product_id") == "P001")
        .select(
            "product_id",
            "catalog_price",
            "updated_at"
        )
    )

    display(catalog_product)

    product_row = catalog_product.first()

    assert product_row is not None, (
        "P001 is missing from the catalogue."
    )

    assert product_row["catalog_price"] == Decimal("27.50"), (
        "P001 should have a current catalogue price of 27.50."
    )

    # Check the order-line values from the same test scenario.
    historical_order_line = (
        spark.table(f"{catalog}.silver.order_items_current")
        .filter(
            (F.col("order_id") == "O10001")
            & (F.col("line_id") == 1)
        )
        .select(
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_amount",
        )
    )

    display(historical_order_line)

    order_line = historical_order_line.first()

    assert order_line is not None, (
        "The test order line is missing."
    )

    assert (
        order_line["product_id"] == "P001"
        and order_line["quantity"] == 2
        and order_line["unit_price"] == Decimal("25.00")
        and order_line["line_amount"] == Decimal("50.00")
    ), "The test order line no longer matches its expected values."

    print("Catalogue update and order-price fixture checks passed.")

else:
    print("Skipped the fixed product-price and order-line fixture tests.")